# Experiment: Qwen2.5-7B induction-copy mechanism surface (Copy-v2)

**Question.** On a prospectively defined clean-eligible prompt population, does Qwen2.5-7B contain a causally validated induction-copy head panel, and do interaction-aware observers improve prediction or action selection on its frozen eight-head intervention surface?

**Success criteria.** The candidate reservoir is fixed before model scoring. Clean eligibility must pass every frozen coverage gate before any attention scan or intervention. A later causal or locked-test null remains a reportable negative result. Copy-v1 stays negative and is never continued.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


## Before running

- Select an A100, H100, or H200 runtime with at least 40 GiB GPU memory.
- Upload the source-sealed archive to `/content/observerbench-qwen-copy-v2-source.tar.gz` and set `OBSERVERBENCH_ARCHIVE_SHA256` to the handoff digest, or put its verified extraction at `/content/ObserverBench`. This notebook contains no repository URL or credential.
- Run from a copy of this notebook. Saving outputs into the source-sealed notebook would change its source hash.
- Keep artifacts on Drive. Resume only within the same frozen source, config, model revision, and compatible runtime.
- Do not bypass a failed gate. The runner exits nonzero and leaves every later intervention or locked outcome unopened.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import tarfile

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

default_repo = Path("/content/ObserverBench") if IN_COLAB else Path.cwd()
REPO_ROOT = Path(os.environ.get("OBSERVERBENCH_REPO", default_repo)).expanduser().resolve()
if IN_COLAB and not (REPO_ROOT / "pyproject.toml").is_file():
    archive = Path(os.environ.get("OBSERVERBENCH_ARCHIVE", "/content/observerbench-qwen-copy-v2-source.tar.gz"))
    expected_archive_sha256 = os.environ.get("OBSERVERBENCH_ARCHIVE_SHA256", "")
    assert archive.is_file() and len(expected_archive_sha256) == 64, "Upload the sealed archive and set its handoff SHA-256."
    observed_archive_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
    assert observed_archive_sha256 == expected_archive_sha256, "Source archive SHA-256 mismatch."
    REPO_ROOT.mkdir(parents=True, exist_ok=False)
    with tarfile.open(archive, mode="r:gz") as bundle:
        bundle.extractall(REPO_ROOT, filter="data")
default_artifacts = (
    Path("/content/drive/MyDrive/ObserverBenchArtifacts/phase10/qwen_induction/copy_v2")
    if IN_COLAB
    else REPO_ROOT / "results/revision/phase10/qwen_induction_copy_v2"
)
ARTIFACTS_ROOT = Path(os.environ.get("OBSERVERBENCH_COPY_V2_ARTIFACTS", default_artifacts)).expanduser().resolve()
CONFIG = REPO_ROOT / "configs/revision/phase10/qwen2_5_7b_induction_copy_v2.json"
SOURCE_MANIFEST = REPO_ROOT / "configs/revision/phase10/qwen_copy_v2_source_manifest.json"
SCRIPT = REPO_ROOT / "scripts/run_qwen_induction_phase10.py"
CONSTRAINTS = REPO_ROOT / "configs/revision/phase10/colab_constraints.txt"

for path in (REPO_ROOT / "pyproject.toml", CONFIG, SOURCE_MANIFEST, SCRIPT, CONSTRAINTS):
    assert path.is_file(), f"Missing frozen input: {path}"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
{"repo_root": str(REPO_ROOT), "artifacts_root": str(ARTIFACTS_ROOT), "in_colab": IN_COLAB}


## Install and verify the frozen study

The notebook stays thin. Prompt construction, clean eligibility, head discovery, interventions, checkpointing, observer freezing, and evaluation live in the source-sealed package.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-c", str(CONSTRAINTS), "-e", f"{REPO_ROOT}[qwen]"],
    cwd=REPO_ROOT,
    check=True,
)

import torch
from observerbench.provenance import json_sha256
from observerbench.tasks.qwen_induction.artifacts import QWEN_INDUCTION_COPY_V2_CONFIG_SHA256

assert torch.cuda.is_available(), "Select a GPU runtime before the scientific run."
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
assert any(name in gpu_name.upper() for name in ("A100", "H100", "H200")) and gpu_memory_gib >= 39

frozen_config = json.loads(CONFIG.read_text())
source_manifest = json.loads(SOURCE_MANIFEST.read_text())
CONFIG_SHA256 = json_sha256(frozen_config)
assert frozen_config["status"] == "frozen_before_copy_v2_outcomes"
assert frozen_config["data_version"] == "copy-v2"
assert CONFIG_SHA256 == QWEN_INDUCTION_COPY_V2_CONFIG_SHA256
assert source_manifest["config_sha256"] == CONFIG_SHA256
{
    "gpu": gpu_name,
    "gpu_memory_gib": round(gpu_memory_gib, 1),
    "torch": torch.__version__,
    "config_sha256": CONFIG_SHA256,
    "source_bundle_sha256": source_manifest["source_bundle_sha256"],
}


## Frozen stage order

1. `prepare`: write the new 2× candidate reservoirs and prove Copy-v1 token exclusion without loading Qwen.
2. `eligibility`: score clean prompts only, apply the fixed `candidate_correct` and `margin ≥ ln(4)` rule, require eager/SDPA agreement for the discovery bank, enforce coverage, and hash-select final prompts.
3. `discover` and `confirm`: identify heads, then test them against matched controls on held-out prompts.
4. `freeze-design`, `measure-calibration`, and `freeze-observers`: freeze the mask design, measurements, predictions, targets, and actions.
5. `measure-locked-test`, `evaluate`, and `measure-collateral`: open outcomes only after the freeze, evaluate, then run the secondary collateral diagnostic.


In [ ]:
def run_stage(stage: str):
    command = [
        sys.executable, str(SCRIPT),
        "--config", str(CONFIG),
        "--artifacts-root", str(ARTIFACTS_ROOT),
        "--stage", stage,
        "--resume",
    ]
    print("Running:", stage)
    return subprocess.run(
        command, cwd=REPO_ROOT, check=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"},
    )


## 1. Freeze candidate reservoirs

This stage performs no model forward pass. It writes fresh prompt banks, proves they exclude every Copy-v1 allocated token, and seals their hashes.


In [ ]:
run_stage("prepare")


## 2. Apply the clean-only eligibility gate

This is the first Qwen forward pass. It cannot scan attention or call an intervention. Failure is terminal for Copy-v2 and leaves the desired second-mechanism claim unavailable.


In [ ]:
try:
    run_stage("eligibility")
finally:
    frozen_coverage = ARTIFACTS_ROOT / "design/eligibility/coverage.csv"
    work_coverage = ARTIFACTS_ROOT / "work/eligibility/coverage.csv"
    coverage = frozen_coverage if frozen_coverage.is_file() else work_coverage
    diagnostics = ARTIFACTS_ROOT / "work/eligibility/coverage_diagnostics.json"
    if coverage.is_file():
        print(coverage.read_text())
    if diagnostics.is_file():
        print(json.dumps(json.loads(diagnostics.read_text()), indent=2)[:12000])


## 3. Discover and independently confirm the head panel

Only the passed clean freeze permits these cells. Discovery uses the selected discovery and head-fit banks. Confirmation uses a disjoint bank and must show that the selected panel impairs copy discrimination more than matched controls.


In [ ]:
run_stage("discover")
run_stage("confirm")


## 4. Freeze the intervention design and measure calibration

A passed confirmation gate freezes the eight heads, full 256-mask Boolean cube, complementary calibration/test split, and action pools. Calibration outcomes alone may then be measured.


In [ ]:
run_stage("freeze-design")
run_stage("measure-calibration")


## 5. Freeze observers, targets, and actions

This stage fits the registered no-effect, additive, and quadratic observers at budgets 16, 40, 64, and 128. It seals every prediction and action before a locked effect can be read.


In [ ]:
locked_candidates = [
    ARTIFACTS_ROOT / "work/measurements/locked_test_effects_raw.csv",
    ARTIFACTS_ROOT / "effects/test_effects.csv",
]
assert not any(path.exists() for path in locked_candidates), "Locked outcomes exist too early."
run_stage("freeze-observers")


## 6. Open the locked surface and evaluate

The primary prediction contrast is additive MAE minus quadratic MAE at 128 calibration masks. Action comparisons retain all three targets, their equal-weight aggregate, and exact no-op. A null or loss to no-op is a negative result, not a reason to change the observer.


In [ ]:
run_stage("measure-locked-test")
run_stage("evaluate")
run_stage("measure-collateral")


## Artifact audit and decision record

This inventory confirms completion only. Interpret the frozen gate files, effect sizes, bootstrap intervals, and action contrasts before changing the manuscript.


In [ ]:
audit = json.loads((ARTIFACTS_ROOT / "work/stage_audit.json").read_text())
artifact_files = sorted(
    path.relative_to(ARTIFACTS_ROOT).as_posix()
    for path in ARTIFACTS_ROOT.rglob("*") if path.is_file()
)
{
    "terminal_status": audit["terminal_status"],
    "completed_stages": audit["completed_stages"],
    "artifact_count": len(artifact_files),
    "last_20_paths": artifact_files[-20:],
}


## How to classify the result

- **Coverage failure:** negative for the registered conditional fixture; no intervention evidence exists.
- **Causal confirmation failure:** no Qwen mechanism-surface claim.
- **Causal pass, locked prediction null:** positive second model/mechanism surface, negative interaction-aware transfer.
- **Quadratic prediction and direct-risk action wins:** positive transfer claims only for the frozen clean-eligible, candidate-constrained population.

Always report Copy-v1's broader clean-gate failure beside Copy-v2.
